# 故障很少时，怎样避免模型只预测正常？：参考答案

每道题先写预期结果再运行。练习中的 TODO 需要自行完成；能顺序运行不代表已完成练习。

In [1]:
from pathlib import Path
CASE_DIR = Path.cwd()
# 载入本例的可修改函数和数据，所有算法都在同目录 solution.py。
exec(compile((CASE_DIR / "solution.py").read_text(encoding="utf-8"), str(CASE_DIR / "solution.py"), "exec"))

{'selected_threshold': 0.060000000000000005, 'all_normal': {'tn': 330, 'fp': 0, 'fn': 30, 'tp': 0, 'cost': 600.0}, 'threshold_0_5': {'tn': 328, 'fp': 2, 'fn': 14, 'tp': 16, 'cost': 282.0}, 'selected': {'tn': 247, 'fp': 83, 'fn': 2, 'tp': 28, 'cost': 123.0}, 'precision': 0.25225225225225223, 'recall': 0.9333333333333333, 'raw_brier': 0.11510529881262979, 'calibrated_brier': 0.038271708162379084}


## 1. 补全：误报漏报成本
用 selected 中的 fp/fn 复算代价。

In [2]:
computed = metrics['selected']['fp'] + 20*metrics['selected']['fn']
assert computed == metrics['selected']['cost']
print(computed)

123


## 2. 修改：漏报成本减半
仅在验证集用漏报成本 10 重新选阈值，报告新阈值而不在测试集选。

In [3]:
new_rows = [(t,error_cost(y_val,p_val>=t,missed=10)['cost']) for t in np.linspace(0.01,0.8,80)]
new_threshold, cost = min(new_rows,key=lambda row:row[1])
assert 0 < new_threshold < 1
print('成本改变后的验证阈值',new_threshold,'成本',cost)

成本改变后的验证阈值 0.060000000000000005 成本 78.0


## 3. 迁移：报警容量只有 15 台
在测试概率上按风险排序最多报警 15 台；用真实标签事后核算，不把标签用于排序。

In [4]:
limited = np.zeros(len(p_test),dtype=bool)
limited[np.argsort(-p_test)[:15]] = True
assert limited.sum() == 15
print(error_cost(y_test,limited))

{'tn': 329, 'fp': 1, 'fn': 16, 'tp': 14, 'cost': 321.0}
